In [1]:
# Install Packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm

In [2]:
# Load in tables
athletes = pd.read_csv("../data/athlete_events.csv")
city_coords = pd.read_csv("../data/city_coords.csv")
country_coords = pd.read_csv("../data/country_coords.csv")
noc_regions = pd.read_csv("../data/noc_regions.csv")
country_temperatures = pd.read_csv("../data/country_temperature_lookup.csv")
city_temperatures = pd.read_csv("../data/city_temperature_lookup.csv")

In [3]:
# Merge all original tables
df = athletes.merge(
    noc_regions,
    on="NOC",
    how="left"
)
df = df.merge(
    country_coords,
    left_on="region",
    right_on="Country",
    how="left"
)
df = df.rename(columns={
    "Latitude": "country_lat",
    "Longitude": "country_lon"
})
df = df.merge(
    city_coords,
    on="City",
    how="left"
)
df = df.rename(columns={
    "Latitude": "host_lat",
    "Longitude": "host_lon"
})
df = df.merge(
    country_temperatures,
    on=["NOC", "Games"],
    how="left"
)
df = df.merge(
    city_temperatures,
    on=["City", "Games"],
    how="left"
)

In [4]:
# Filter out rows before 1950 (no temperature data)
df = df[df["Year"] >= 1950]

# Remove non country NOCs
df = df[~df["NOC"].isin(["IOA", "ROT"])]

In [5]:
# Check that all entries have temperature data
print(df["avg_temp (NOC)"].isna().sum())
print(df["avg_temp (City)"].isna().sum())

0
0


In [7]:
# Calculate approximate distance between country and host city
df["distance_to_host"] = np.sqrt(
    (69 * (df["country_lat"] - df["host_lat"]))**2 +
    (69 * np.cos(np.radians(df["host_lat"])) * (df["country_lon"] - df["host_lon"]))**2
)
# df.head()

In [8]:
# Calculate other potential features
df["lat_diff"] = df["country_lat"] - df["host_lat"]
df["lon_diff"] = df["country_lon"] - df["host_lon"]
df["temp_diff"] = df["avg_temp (NOC)"] - df["avg_temp (City)"]

df["is_host"] = (df["region"] == df["Host Country"]).astype(int)

# df.head(50)

In [9]:
# set weighted medal score
df["Medal"] = df["Medal"].fillna("None")
df["medal_score"] = df["Medal"].map({
    "Gold": 5,
    "Silver": 3,
    "Bronze": 1,
    "None": 0
})
# df.head()

In [10]:
# Group by NOC, medal, games, and event to remove duplicate entries for team sports
event_results = (
    df.groupby(["NOC","Games","Event"], as_index=False)
    .agg(
        Medal=("Medal","first"),
        medal_score=("medal_score","first"),
        distance_to_host=("distance_to_host","first"),
        lat_diff=("lat_diff","first"),
        lon_diff=("lon_diff","first"),
        temp_diff=("temp_diff","first"),
        is_host=("is_host","first")
    )
)
event_results["won_medal"] = event_results["Medal"] != "None"

# Expect 100472 from sqlite query
# print(len(event_results))
# event_results.head()

In [11]:
country_overall = (
    event_results
    .groupby(["NOC"], as_index=False)
    .agg(
        medals=("won_medal", "sum"),
        medal_score=("medal_score", "sum"),
        events_participated=("Event", "size"),
        distance_to_host=("distance_to_host", "first")
    )
)
country_overall["avg_medals_per_event"] = (
    country_overall["medals"] /
    country_overall["events_participated"]
)

country_overall["avg_score_per_event"] = (
    country_overall["medal_score"] /
    country_overall["events_participated"]
)

# country_overall.head()

In [12]:
country_games = (
    event_results
    .groupby(["NOC","Games"], as_index=False)
    .agg(
        medals=("won_medal","sum"),
        medal_score=("medal_score","sum"),
        events_participated=("Event","size"),

        distance_to_host=("distance_to_host","first"),
        lat_diff=("lat_diff","first"),
        lon_diff=("lon_diff","first"),
        temp_diff=("temp_diff","first"),
        is_host=("is_host","first")
    )
)

country_games["medals_per_event"] = (
    country_games["medals"] /
    country_games["events_participated"]
)

country_games["score_per_event"] = (
    country_games["medal_score"] /
    country_games["events_participated"]
)
country_games = country_games.merge(
    country_overall[["NOC","avg_medals_per_event","avg_score_per_event"]],
    on="NOC",
    how="left"
)
# country_games.head()

In [15]:
X1 = country_games[["distance_to_host", "lat_diff", "lon_diff", "temp_diff", "is_host"]]
X1 = sm.add_constant(X)
Y1 = country_games["score_per_event"]
model = sm.OLS(Y1, X1).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:        score_per_event   R-squared:                       0.086
Model:                            OLS   Adj. R-squared:                  0.084
Method:                 Least Squares   F-statistic:                     61.84
Date:                Fri, 08 May 2026   Prob (F-statistic):           8.36e-62
Time:                        20:19:02   Log-Likelihood:                 232.56
No. Observations:                3309   AIC:                            -453.1
Df Residuals:                    3303   BIC:                            -416.5
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.1315      0.007  

In [16]:
X2 = country_games[["distance_to_host"]]
X2 = sm.add_constant(X2)
Y2 = country_games["score_per_event"]
model = sm.OLS(Y2, X2).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:        score_per_event   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.9238
Date:                Fri, 08 May 2026   Prob (F-statistic):              0.337
Time:                        20:19:04   Log-Likelihood:                 84.962
No. Observations:                3309   AIC:                            -165.9
Df Residuals:                    3307   BIC:                            -153.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                0.1235      0.007  

In [ ]:
X3 = country_games[["lat_diff", "lon_diff", "temp_diff", "is_host"]]
X3 = sm.add_constant(X3)
Y3 = country_games["score_per_event"]
model = sm.OLS(Y3, X3).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:        score_per_event   R-squared:                       0.086
Model:                            OLS   Adj. R-squared:                  0.084
Method:                 Least Squares   F-statistic:                     77.32
Date:                Fri, 08 May 2026   Prob (F-statistic):           8.96e-63
Time:                        20:19:06   Log-Likelihood:                 232.55
No. Observations:                3309   AIC:                            -455.1
Df Residuals:                    3304   BIC:                            -424.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1322      0.004     32.194      0.0

: 